In [1]:
import time

start_time = time.perf_counter()

# IMPORT API KEY

In [2]:
import getpass
import os

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

# SEARCH TOOL DEFINITION

In [3]:
from pydantic import BaseModel, Field
from typing import List, Literal, Optional

class TavilySearchInput(BaseModel):
    """Input parameters for the Tavily web search tool."""
    query: str = Field(description="The search query string.")
    max_results: int = Field(default=5, description="Maximum number of search results to return.")
    topic: Literal["general", "news", "finance"] = Field(default="general", description="Category of the search.")
    include_answer: bool = Field(default=False, description="Include a short answer to the original query in results.")
    include_raw_content: bool = Field(default=False, description="Include cleaned and parsed HTML of search results.")
    include_images: bool = Field(default=False, description="Include a list of query-related images.")
    include_image_descriptions: bool = Field(default=False, description="Include descriptive text for each image.")
    search_depth: Literal["basic", "advanced"] = Field(default="basic", description="Depth of the search.")
    time_range: str = Field(default="", description="Time range to filter results ('day', 'week', 'month', 'year') or empty string.")
    include_domains: List[str] = Field(default_factory=list, description="List of domains to specifically include.")
    exclude_domains: List[str] = Field(default_factory=list, description="List of domains to specifically exclude.")
    country: str = Field(default="", description="Country to focus the search on.")


In [4]:
from langchain.tools import tool
from langchain_tavily import TavilySearch
from typing import List, Optional

@tool(args_schema=TavilySearchInput)
def tavily_search(
    query: str,
    max_results: int = 5,
    topic: str = "general",
    include_answer: bool = False,
    include_raw_content: bool = False,
    include_images: bool = False,
    include_image_descriptions: bool = False,
    search_depth: str = "basic",
    time_range: str = "",
    include_domains: Optional[List[str]] = None,
    exclude_domains: Optional[List[str]] = None,
    country: str = ""
) -> str:
    """
    Perform a web search using TavilySearch.

    Args:
        query: The search query string.
        max_results: Maximum number of search results to return.
        topic: Category of the search ("general", "news", "finance").
        include_answer: Include an answer to the original query in results.
        include_raw_content: Include cleaned and parsed HTML of each search result.
        include_images: Include a list of query-related images in the response.
        include_image_descriptions: Include descriptive text for each image.
        search_depth: Depth of the search ("basic" or "advanced").
        time_range: Time range to filter results ("day", "week", "month", "year") or "".
        include_domains: List of domains to specifically include.
        exclude_domains: List of domains to specifically exclude.
        country: Country to focus the search on.

    Returns:
        Search results as a string.
    """
    
    # parameters validation (Tavily needs None values)
    _time_range = None if not time_range else time_range
    _include_domains = None if not include_domains else include_domains
    _exclude_domains = None if not exclude_domains else exclude_domains
    _country = None if not country else country

    tool_instance = TavilySearch(
        max_results=max_results,
        topic=topic,
        include_answer=include_answer,
        include_raw_content=include_raw_content,
        include_images=include_images,
        include_image_descriptions=include_image_descriptions,
        search_depth=search_depth,
        time_range=_time_range,
        include_domains=_include_domains,
        exclude_domains=_exclude_domains,
        country=_country
    )

    return tool_instance.invoke({"query": query})

# EXTRACT TOOL DEFINITION

In [5]:
from pydantic import BaseModel, Field
from typing import List, Literal

class TavilyExtractInput(BaseModel):
    """Input parameters for the Tavily web page extraction tool."""
    urls: List[str] = Field(description="List of URLs to extract content from.")
    extract_depth: Literal["basic", "advanced"] = Field(default="basic", description="Extraction depth.")
    chunks_per_source: Literal[1, 2, 3, 4, 5] = Field(default=3, description="Number of text chunks to extract per source (max 5).")
    include_images: bool = Field(default=False, description="Whether to include images in the extraction.")
    include_favicon: bool = Field(default=False, description="Whether to include favicons in the extraction.")
    format_text: Literal["markdown", "text"] = Field(default="markdown", description="Output formatting type.")


In [6]:
from langchain.tools import tool
from langchain_tavily import TavilyExtract

@tool(args_schema=TavilyExtractInput)
def tavily_extract(
    urls: List[str],
    extract_depth: str = "basic",
    chunks_per_source: int = 3,
    include_images: bool = False,
    include_favicon: bool = False,
    format_text: str = "markdown"
) -> str:
    """
    Extract content from a web page using TavilyExtract.

    Args:
        urls: The URL list of the pages to extract content from.
        extract_depth: Extraction depth ("basic" or "advanced").
        chunks_per_source: Number of chunks per source (1-5).
        include_images: Whether to include images (True or False).
        include_favicon: Whether to include favicon (True or False).
        format_text: Output format ("markdown" or "text").

    Returns:
        Extracted content as a string.
    """
    tool_instance = TavilyExtract(
        extract_depth=extract_depth,
        chunks_per_source=chunks_per_source,
        include_images=include_images,
        include_favicon=include_favicon,
        format=format_text,
    )

    return tool_instance.invoke({"urls": urls})

# RETRIEVE TOOL DEFINITION

In [7]:
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

# AGENT DEFINITION

In [8]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b")

agent = create_agent(
    model=model,
    tools=[tavily_search, tavily_extract, retrieve_context],
    checkpointer=InMemorySaver(),
    # system_prompt = """
    # You are an advanced web research and conversational RAG assistant. You have three specific tools at your disposal. You must choose the appropriate one based on the user's request.
    
    # 1. WEB SEARCH: 
    # If the user provides a query with some parameters, use the 'tavily_search' tool.
    # - The user's input will consist of two parts: 'text' (the query) and 'params' (a dictionary of tool parameters).
    # - Only use the parameters provided by the user. Execute the tool using exactly the parameters given.
    
    # 2. WEBPAGE EXTRACTION: 
    # If the user provides a list of URLs, use the 'tavily_extract' tool.
    # - Retrieve the entire original content from the webpage without any modifications, adjustments or alterations whatsoever (no changes to the lexicon, punctuation, format, or language).
    # - REPORTING RULES (CRITICAL): Your final response MUST be a verbatim report of the extracted content. Do NOT modify the content, summarize it, or add introductory/concluding text.
    # - DO NOT add any AI-generated headers or markdown structures not present in the original tool output.
    # - Format each source in the 'tavily_extract' tool result exactly like this:
    #     ----- SOURCE: <url>
    #     <raw_content>
    # - Separate different sources with TWO NEWLINES. Example of two results retrieved from the <tavily_extract> tool:

    #     ----- SOURCE: <url_1>
    #     <raw_content_1>


    #     ----- SOURCE: <url_2>
    #     <raw_content_2>
        
    # - Strictly use these parameters: extract_depth="basic", chunks_per_source=3, include_images=False, include_favicon=False, format_text="markdown".
    
    # 3. CONVERSATIONAL RAG & QUESTION ANSWERING: 
    # If the user asks a natural language question or wants to discuss a specific topic, use the 'retrieve_context' tool.
    # - Query the vector store to retrieve relevant context.
    # - CRITICAL RULE: You must base your answer *EXCLUSIVELY* on the retrieved context.
    # - If the answer to the user's question is not present in the retrieved context, state clearly that you do not have enough information in the extracted sources. DO NOT guess, synthesize outside knowledge, or hallucinate.
    # - Maintain a helpful, clear, and conversational tone while answering in the user's language.
    # """,
    system_prompt = """
    You are an advanced web research and conversational RAG assistant. You have three specific tools at your disposal. 
    You must choose the appropriate one based on the user's request.
    
    1. WEB SEARCH: 
    If the user provides a query alongside a dictionary of parameters, use the 'tavily_search' tool.
    - The user's input will consist of two parts, usually formatted like: 'text: <query>, params: <json_dictionary>'.
    - Extract the parameters from the 'params' dictionary and pass them EXACTLY as they are into the 'tavily_search' tool 
    schema. Do not change or invent parameters.
    - Use the 'text' input as the 'query' field for the tool.
      
    2. WEBPAGE EXTRACTION:
    If the user provides a list of URLs to extract, use the 'tavily_extract' tool.
    - Pass ONLY the 'urls' to the tool. Do not attempt to specify any other optional parameter, rely entirely on the tool's 
    default settings.
    - Retrieve the entire original content from the webpage without any modifications, adjustments or alterations whatsoever 
    (no changes to the lexicon, punctuation, format, or language).
    - REPORTING RULES (CRITICAL): Your final response MUST be a verbatim report of the extracted content. Do NOT modify the content, 
    summarize it, or add any introductory/concluding text.
    - DO NOT add any AI-generated headers or markdown structures not present in the original tool output.
    - Format each source in the result exactly like this:
        ----- SOURCE: <url>
        
        <raw_content>

        
    - Separate different sources with TWO NEWLINES. Example of two results retrieved from the <tavily_extract> tool:

        ---------- SOURCE: <url_1>
        
        <raw_content_1>


        ---------- SOURCE: <url_2>
        
        <raw_content_2>

    
    3. CONVERSATIONAL RAG & QUESTION ANSWERING: 
    If the user asks a natural language question or wants to discuss a specific topic, use the 'retrieve_context' tool.
    - Query the vector store to retrieve relevant context.
    - CRITICAL RULE: You must base your answer *EXCLUSIVELY* on the retrieved context.
    - If the answer to the user's question is not present in the retrieved context, state clearly that you do not have enough 
    information in the extracted sources. DO NOT guess, synthesize outside knowledge, or hallucinate.
    - Maintain a helpful, clear, and conversational tone while answering in the user's language.
    """,
)

# WEB SEARCH INVOCATION

In [9]:
import json

default_params = {    
    'max_results': 5,
    'topic': "general",
    'include_answer': False,
    'include_raw_content': False,
    'include_images': False,
    'include_image_descriptions': False,
    'search_depth': "basic",
    'time_range': "",
    'include_domains': [],
    'exclude_domains': [],
    'country': "",
}
user_ui_params = {"include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}
params = {**default_params, **user_ui_params}
params_json = json.dumps(params)

research_query = "\"Pasta alla carbonara\""
query = f"text: {research_query}, params: {params_json}"

result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    {"configurable": {"thread_id": "1"}},
)

###### NOT STRUCTURED RESULT

In [10]:
result

{'messages': [HumanMessage(content='text: "Pasta alla carbonara", params: {"max_results": 5, "topic": "general", "include_answer": false, "include_raw_content": false, "include_images": false, "include_image_descriptions": false, "search_depth": "basic", "time_range": "", "include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}', additional_kwargs={}, response_metadata={}, id='bbda1e3c-4e87-4536-851f-69579aea27e9'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-12T14:12:26.2921501Z', 'done': True, 'done_reason': 'stop', 'total_duration': 169013611100, 'load_duration': 9507429800, 'prompt_eval_count': 1736, 'prompt_eval_duration': 67192556400, 'eval_count': 261, 'eval_duration': 91967473400, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--019d8206-cdf9-7d03-a838-6146fd180908-0', tool_calls=[{'name': 'tavily_search', 'args': {'include_raw_content': False, 'include_

### Results filtering

In [11]:
overview = result["messages"][-1]
overview.pretty_print()

================================== Ai Message ==================================

Ecco le informazioni sulla **Pasta alla Carbonara** da 5 siti web italiani:

---

## 1. Roma Hotel Cressy
**URL:** https://www.romahotelcressy.it/la-pasta-alla-carbonara/

La pasta alla carbonara è un piatto tipico della cucina romana, ricco di storia e di varianti che si sono sviluppate nel corso del tempo. La sua origine esatta è ancora oggetto di dibattito, ma diverse teorie si concentrano sull'influenza delle razioni militari portate dagli eserciti alleati durante la Seconda Guerra Mondiale, che comprendevano uova e bacon, combinati con la pasta italiana. La carbonara ha anche suscitato interesse nella cultura popolare, con menzioni in canzoni e film, e addirittura con la celebrazione del "Carbonara Day" il 6 aprile di ogni anno, un evento sociale che celebra questa iconica ricetta italiana. La base della carbonara rimane comunque costituita da uova, pancetta o guanciale e formaggio pecorino.

---

##

In [12]:
import json

tool_msg = json.loads(result["messages"][-2].content)

research_results = tool_msg["results"]
research_results

[{'url': 'https://www.romahotelcressy.it/la-pasta-alla-carbonara/',
  'title': 'LA PASTA ALLA CARBONARA - Roma Hotel CressyRoma Hotel Cressy',
  'content': 'La pasta alla carbonara è un piatto tipico della cucina romana, ricco di storia e di varianti che si sono sviluppate nel corso del tempo. La sua origine esatta è ancora oggetto di dibattito, ma diverse teorie si concentrano sull’influenza delle razioni militari portate dagli eserciti alleati durante la Seconda Guerra Mondiale, che comprendevano uova e bacon, combinati con la pasta italiana. La carbonara ha anche suscitato interesse nella cultura popolare, con menzioni in canzoni e film, e addirittura con la celebrazione del “Carbonara Day” il 6 aprile di ogni anno, un evento sociale che celebra questa iconica ricetta italiana. La base della carbonara rimane comunque costituita da uova, pancetta o guanciale e formaggio pecorino. Aggiungere anche un po’ di acqua di cottura della pasta non salata per rendere la crema più morbida. Togl

# WEB PAGE EXTRACTION INVOCATION

In [13]:
from langchain.messages import HumanMessage

query = HumanMessage(content=[
    {"type": "text", "text": [research_results[0]["url"], research_results[-1]["url"]]},
])

response = agent.invoke(
    {"messages": [query]},
    {"configurable": {"thread_id": "1"}},
)

In [14]:
# DISPLAY TOOL MESSAGE
from IPython.display import Markdown, display

display(Markdown(response["messages"][-2].content))

{"results": [{"url": "https://www.romahotelcressy.it/la-pasta-alla-carbonara/", "title": "CARBONARA PASTA - Roma Hotel Cressy", "raw_content": "![it](https://www.romahotelcressy.it/wp-content/plugins/sitepress-multilingual-cms/res/flags/it.png)\n![en](https://www.romahotelcressy.it/wp-content/plugins/sitepress-multilingual-cms/res/flags/en.png)\n![Roma Hotel Cressy](https://www.romahotelcressy.it/wp-content/uploads/2020/09/HC-h95.jpg)\n\n### Main menu\n\n### LA PASTA ALLA CARBONARA\n\n![](https://www.romahotelcressy.it/wp-content/uploads/2024/04/Espaguetis_carbonara.jpeg)\n\n# LA PASTA ALLA CARBONARA\n\nLA STORIA\n\nLa pasta alla carbonara è un piatto tipico della cucina romana, ricco di storia e di varianti che si sono sviluppate nel corso del tempo.  \nLa sua origine esatta è ancora oggetto di dibattito, ma diverse teorie si concentrano sull’influenza delle razioni militari portate dagli eserciti alleati durante la Seconda Guerra Mondiale, che comprendevano uova e bacon, combinati con la pasta italiana.  \nTra le varie ipotesi, vi è quella anglosassone che suggerisce che i soldati americani durante l’occupazione di Roma abbiano combinato gli ingredienti a loro disposizione, dando così origine alla ricetta. Altre teorie attribuiscono l’origine della carbonara alla cucina napoletana o all’Appennino abruzzese, con ingredienti come cacio e uova.  \nNel corso degli anni, la ricetta si è evoluta e sono emerse numerose varianti, sia in Italia che all’estero. Alcune di queste varianti includono l’uso di formaggi diversi, l’aggiunta di panna, cipolle, funghi o sostituzioni di ingredienti per adattarsi a particolari restrizioni dietetiche, come nel caso delle versioni kosher.  \nLa carbonara ha anche suscitato interesse nella cultura popolare, con menzioni in canzoni e film, e addirittura con la celebrazione del “Carbonara Day” il 6 aprile di ogni anno, un evento sociale che celebra questa iconica ricetta italiana.\n\n**INGREDIENTI**  \nLa base della carbonara rimane comunque costituita da uova, pancetta o guanciale e formaggio pecorino.\n\nPer 4 persone\n\n1.Spaghetti 320 g.  \n2.Tuorli 6  \n3.Pepe nero q.b.  \n4.Guanciale 150 g.  \n5.Pecorino romano 50 g.\n\n**PREPARAZIONE**  \nRimuovere la cotenna e tagliare il guanciale a listarelle spesse. Metterlo in una padella a fuoco dolce fino a quando il grasso diventa lucido.   \nLavorare i tuorli con il pecorino e il pepe nero fino a ottenere una crema. Aggiungere un po’ del grasso del guanciale e mescolare. Aggiungere anche un po’ di acqua di cottura della pasta non salata per rendere la crema più morbida.   \nCuocere la pasta in acqua non troppo salata fino a quando è al dente. Scolarla, mantenendo un po’ di acqua di cottura nella padella con il guanciale.   \nTogliere il guanciale dalla padella e aggiungere la pasta scolata. Unire la crema di tuorli e formaggio alla pasta e amalgamare bene a fuoco spento. Aggiungere il guanciale sopra la pasta una volta impiattata.   \nCompleta il piatto con pecorino e pepe nero macinato fresco.   \nSeguendo questi passaggi, otterrai una deliziosa pasta alla carbonara cremosa e piena di sapore. Buon appetito!\n\nCREDITS FOTO  \nJavier Somoza, [CC BY-SA 4.0, via Wikimedia Commons](https://creativecommons.org/licenses/by-sa/4.0)\n\nPrenota il tuo soggiorno a Roma in maniera diretta **risparmia sulle commissioni**\n\n## TEL: +39 06 48 49 17 hotelcressy@gmail.com\n\n[![](https://www.romahotelcressy.it/wp-content/uploads/2023/05/WA-150x150.jpg)](https://api.whatsapp.com/send?phone=393517634333&text=Salve%20Hotel%20Cressi%20Roma,%20vorrei%20chattare%20con%20voi)\n\n![](https://www.romahotelcressy.it/wp-content/uploads/2023/05/WA-150x150.jpg)\n\nComments are closed.\n\n### Search\n\n### Recent Posts\n\n![](https://www.romahotelcressy.it/wp-content/uploads/2015/06/TripAdvisor-logo-300x51.png)", "images": []}, {"url": "https://www.ceramicavietrese.it/en/recipes-claudia/a-typical-italian-food--pasta-alla-carbonara-a-typical-recipe-of-rome-tradition-and-italian-tradition/", "title": "Pasta alla \"Carbonara\" a typical recipe of Rome tradition and Italian ...", "raw_content": "[![VIETRI CERAMICS - the excellence artisan pottery made in Italy]()  VIETRI CERAMICS - the excellence artisan pottery made in Italy](/en/) \n\n[0](/en/cart/)  [Login](/en/login/)\n\n* [Italiano](/)\n* [English](/en/)\n* [Español](/es/)\n* [Deutsch](/de/)\n* [Français](/fr/)\n* [Pусский](/ru/)\n* [中国](/c2/)\n\n* [Home](/en/)\n* [About us](#) \n  + [Vietri and the ceramic](/en/vietri-and-the-ceramic/)\n  + [Our master potters](/en/our-master-potters/)\n  + [The art of manufacturing ceramic](/en/the-art-of-manufacturing-ceramic/)\n  + [Our design furniture](/en/our-design-furniture/)\n  + [Photo Gallery](/en/photo-gallery/)\n  + [Video-Gallery](/en/video-gallery/)\n  + [Shipments](/en/shipments/)\n  + [Payment methods](/en/payment-methods/)\n  + [Warranty, defects and refunds](/en/warranty-defects-and-refunds/)\n  + [Sure because we have MOCA certification](/en/sure-because-we-have-moca-certification/)\n  + [FAQ Frequently Asked Questions](/en/faq-frequently-asked-questions/)\n  + [Latest News](/en/news/)\n  + [Recipes Claudia](/en/recipes-claudia/)\n* [Products Products](#) \n  + [Complete catalog](/en/products/)\n\n    [Prompt Delivery](/en/products/?filter=stock)\n\n    [Complete catalog](/en/products/)  [Prompt Delivery](/en/products/?filter=stock)\n\n     [![Spice Jar with Royal House Decor 2]()](/en/products/spice-jar-with-royal-house-decor-2/?cat=ceramic-jars)\n\n    ### [On the table and in the kitchen](/en/products/on-the-table-and-in-the-kitchen/)\n\n    - [Dishes](/en/products/dishes/)\n    - [Chef and Gourmet Shapes Dishes](/en/products/chef-and-gourmet-shapes-dishes/)\n    - [PizzaPiatto®](/en/products/pizzapiatto/)\n    - [Breakfast](/en/products/breakfast/)\n    - [Lunch Dinner](/en/products/lunch-dinner/)\n    - [Bowls, Trays and Serving Plates](/en/products/bowls-trays-and-serving-plates/)\n    - [Center Table](/en/products/center-table/)\n    - [Accessories](/en/products/accessories/)\n    - [Ceramic jars](/en/products/ceramic-jars/)\n    - [Terrine da Forno](/en/products/terrine-da-forno/)\n    - [Trays in wood and tiles](/en/products/trays-in-wood-and-tiles/)\n\n    ### [Wedding favors](/en/products/wedding-favors/)\n\n    - [Wedding](/en/products/wedding/)\n    - [Baptism](/en/products/baptism/)\n    - [Birthday, Anniversary, Graduation](/en/products/birthday-anniversary-graduation/)\n    - [First Holy Communion](/en/products/first-holy-communion/)\n    - [Magnets](/en/products/magnets/)\n\n    ### [Home furnishing](/en/products/home-furnishing/)\n\n    - [Bathroom furniture](/en/products/bathroom-furniture/)\n    - [Vasi e Cachepot](/en/products/vasi-e-cachepot/)\n    - [Wall Plates](/en/products/wall-plates/)\n    - [Umbrella stand](/en/products/umbrella-stand/)\n    - [Lumi e Lampshade](/en/products/lumi-e-lampshade/)\n    - [Objects](/en/products/objects/)\n    - [Tiles, Tiles, Floors, Coverings and Kitchen Countertops](/en/products/tiles-tiles-floors-coverings-and-kitchen-countertops/)\n    - [Panels - Murals made of project](/en/products/panels--murals-made-of-project/)\n    - [Tables and Coffee Tables](/en/products/tables-and-coffee-tables/)\n\n    ### [Garden furniture](/en/products/garden-furniture/)\n\n    - [Kitchen Tables, Coffee Tables, Bench and Outdoor Fountain](/en/products/kitchen-tables-coffee-tables-bench-and-outdoor-fountain/)\n    - [Chairs and Stools](/en/products/chairs-and-stools/)\n    - [Fountains](/en/products/fountains/)\n\n    ### [Design furniture](/en/products/design-furniture/)\n\n    - [Bar tables and stools](/en/products/bar-tables-and-stools/)\n    - [Tables and Coffee Tables](/en/products/tables-and-coffee-tables1/)\n    - [Benches and Benches](/en/products/benches-and-benches/)\n\n    ### [Festivity](/en/products/festivity/)\n\n    - [Christmas](/en/products/christmas/)\n    - [Easter](/en/products/easter/)\n\n    ### [Artistic](/en/products/artistic/)\n\n    In this section you will find High Ceramics objects, UNIQUE PIECES created by our. Master ...\n\n    ### [Outlet - Discounts up to 60%](/en/products/outlet-discounts-up-to-60/)\n\n    Sales: Here you can find items DISCOUNTED up to 60% end of series or ex display or with small ...\n* [Gift Card](/en/gift-card/)\n* [Lists  Gifts](#) \n  + [Wedding list](/en/lists-online/wedding-list/)\n  + [Gifts list](/en/lists-online/gifts-list/)\n  + [Create a list](/en/login/?from=lc)\n  + [Search a wedding list](/en/search-for-a-gift-list/?cat=wedding-list)\n  + [Search for a gift list](/en/search-for-a-gift-list/?cat=gifts-list)\n  + [Give a Gift Card](/en/gift-card/)\n* [Contacts](/en/contacts/)\n\n* [Italiano](/)\n* [English](/en/)\n* [Español](/es/)\n* [Deutsch](/de/)\n* [Français](/fr/)\n* [Pусский](/ru/)\n* [中国](/c2/)\n\n[0](/en/cart/ \"Cart\")\n\n* [Home](/en/)\n* [Recipes Claudia](/en/recipes-claudia/)\n* A typical Italian Food - ...\n\n## Recipes Claudia\n\n[Back to the list](/en/recipes-claudia/)\n\n# A typical Italian Food - Pasta alla \"Carbonara\" a typical recipe of Rome tradition and Italian tradition\n\nA typical recipe of Italian traditions , especially Roman traditions . Easy to do, but that once tasted you can hardly forget . Tradition would like to use only seasoned and grated Pecorino Romano, but if you do not like the taste too strong , you can replace a portion of cheese with Original Parmigiano Reggiano (or original Grana Padano ) that definitely has a taste and a flavor less \"strong \" .\n\n[![image](/files/admin/immagini/pasta-alla-carbonara-ricetta-tipica-in-ceramica-di-vietri-ceramica-vietrese4-l-20200910230002.jpg)](/files/admin/immagini/pasta-alla-carbonara-ricetta-tipica-in-ceramica-di-vietri-ceramica-vietrese4-l-20200910230002.jpg \"Pork cheek\") \n\nPork cheek\n\nRecipe for 4 people\n\n***Ingredients:***\n\n- 350 gr. of spaghetti of Gragnano\n\n- 3 eggs + 1 egg yolk\n\n- 160gr of cheek lard\n\n- Roman pecorino\n\n- Roman parmisan\n\n- salt and pepper q.b.\n\n***Preparazione:***\n\nPut a pot with abundant water on the fire and when it is going to boile, you'll salt. Use less salt as usual because pasta have already a tasty condiment with cheek lard, parmesan and pecorino.\n\nWhile you cut the cheek lard in cubes or strips, put in a pot on fire until the fatty become transparent and lightly crounch, then put off from the fire and let it lightly cold.\n\nBreak the eggs in a bowl and mix them. Add the pecorino, parmesan and the pepper (as you like) and mix to obtain an homogeneus mixture. Add the cheek lard.\n\nPut the pasta to cook.\n\nWhen pasta will be ready, dry it and put in the bowl and mix with the mixture and amalgamate them.\n\n***Serve Spaghetti alla Carbonara immediately, adding a fistful of pecorino and black pepper.... and if you used the dish of CeramicaVietrese.it you'll se that they will be better if they are accompanied  with art and colors of Vietri ceramic!!!***\n\n[![MeTour.it](/tpl/default/assets/images/metour-banner-2025.jpg)](https://www.metour.it/it)\n\n##### Vietri Ceramic\n\n* [Vietri and the ceramic](/en/vietri-and-the-ceramic/)\n* [Our master potters](/en/our-master-potters/)\n* [The art of manufacturing ceramic](/en/the-art-of-manufacturing-ceramic/)\n* [Our design furniture](/en/our-design-furniture/)\n* [Photo Gallery](/en/photo-gallery/)\n* [Video-Gallery](/en/video-gallery/)\n* [Latest News](/en/news/)\n* [Recipes Claudia](/en/recipes-claudia/)\n* [Partner Area](/en/login-partner/)\n\n##### Our Shop\n\n* [Products Catalogue](/en/products/)\n* [Shipments](/en/shipments/)\n* [Payment methods](/en/payment-methods/)\n* [Warranty, defects and refunds](/en/warranty-defects-and-refunds/)\n* [Sure because we have MOCA certification](/en/sure-because-we-have-moca-certification/)\n* [FAQ Frequently Asked Questions](/en/faq-frequently-asked-questions/)\n\n##### Lists online\n\n* [Wedding list](/en/lists-online/wedding-list/)\n* [Gifts list](/en/lists-online/gifts-list/)\n* [Create a list](/en/login/?from=lc)\n* [Search a wedding list](/en/search-for-a-gift-list/?cat=wedding-list)\n* [Search for a gift list](/en/search-for-a-gift-list/?cat=gifts-list)\n\n##### Gift Card\n\n* [Give a Gift Card](/en/gift-card/)\n* [Ceramica Vietrese Card](/en/gift-card/ceramica-vietrese-card/)\n* [Mother's Day Card](/en/gift-card/mothers-day-card/)\n* [Happy Birthday Card](/en/gift-card/happy-birthday-card/)\n* [Merry Christmas Card](/en/gift-card/merry-christmas-card/)\n* [Happy Easter Card](/en/gift-card/happy-easter-card/)\n* [Wedding Gift Card](/en/gift-card/wedding-gift-card/)\n\n##### Customer Area\n\n* [User Login](/en/login/)\n* [Not registered?](/en/register-on-our-site/)\n\n##### Informations\n\n* [Terms of sale](/en/terms-of-sale/)\n* [Customer rights](/en/customer-rights/)\n* [Legal notes](/en/legal-notes/)\n* [Cookie policy](/en/cookie-policy/)\n* [Privacy policy](/en/privacy-policy/)\n\n##### Follow us on\n\n[![VK]()](http://vk.com/id344442374)\n\n##### Subscribe our newsletter\n\nYou will receive all updated news directly through your email account.\n\n##### Contacts\n\nCeramica Vietrese di  \n Di Palma Claudia  \n VAT 09299191214\n\n [Contact us](/en/contacts/)\n\n[![Italiano]( \"Italiano\")](/)  [![English]( \"English\")](/en)  [![Español]( \"Español\")](/es)  [![Deutsch]( \"Deutsch\")](/de)  [![Français]( \"Français\")](/fr)  [![Pусский]( \"Pусский\")](/ru)  [![中国]( \"中国\")](/c2)\n\n© 2026 Di Palma Claudia   2015 website by [dynform.it](http://www.dynform.it/)       \n& [studio150.it](http://www.studio150.it/)", "images": []}], "failed_results": [], "response_time": 0.01, "request_id": "73440b10-68bc-4d8e-851c-1a0d7997d904"}

In [15]:
# DISPLAY AI MESSAGE
from IPython.display import Markdown, display

display(Markdown(response["messages"][-1].content))

--- SOURCE: https://www.romahotelcressy.it/la-pasta-alla-carbonara/

![it](https://www.romahotelcressy.it/wp-content/plugins/sitepress-multilingual-cms/res/flags/it.png)
![en](https://www.romahotelcressy.it/wp-content/plugins/sitepress-multilingual-cms/res/flags/en.png)
![Roma Hotel Cressy](https://www.romahotelcressy.it/wp-content/uploads/2020/09/HC-h95.jpg)

### Main menu

### LA PASTA ALLA CARBONARA

![](https://www.romahotelcressy.it/wp-content/uploads/2024/04/Espaguetis_carbonara.jpeg)

# LA PASTA ALLA CARBONARA

LA STORIA

La pasta alla carbonara è un piatto tipico della cucina romana, ricco di storia e di varianti che si sono sviluppate nel corso del tempo.  
La sua origine esatta è ancora oggetto di dibattito, ma diverse teorie si concentrano sull'influenza delle razioni militari portate dagli eserciti alleati durante la Seconda Guerra Mondiale, che comprendevano uova e bacon, combinati con la pasta italiana.  
Tra le varie ipotesi, vi è quella anglosassone che suggerisce che i soldati americani durante l'occupazione di Roma abbiano combinato gli ingredienti a loro disposizione, dando così origine alla ricetta. Altre teorie attribuiscono l'origine della carbonara alla cucina napoletana o all'Appennino abruzzese, con ingredienti come cacio e uova.  
Nel corso degli anni, la ricetta si è evoluta e sono emerse numerose varianti, sia in Italia che all'estero. Alcune di queste varianti includono l'uso di formaggi diversi, l'aggiunta di panna, cipolle, funghi o sostituzioni di ingredienti per adattarsi a particolari restrizioni dietetiche, come nel caso delle versioni kosher.  
La carbonara ha anche suscitato interesse nella cultura popolare, con menzioni in canzoni e film, e addirittura con la celebrazione del "Carbonara Day" il 6 aprile di ogni anno, un evento sociale che celebra questa iconica ricetta italiana.

**INGREDIENTI**  
La base della carbonara rimane comunque costituita da uova, pancetta o guanciale e formaggio pecorino.

Per 4 persone

1.Spaghetti 320 g.  
2.Tuorli 6  
3.Pepe nero q.b.  
4.Guanciale 150 g.  
5.Pecorino romano 50 g.

**PREPARAZIONE**  
Rimuovere la cotenna e tagliare il guanciale a listarelle spesse. Metterlo in una padella a fuoco dolce fino a quando il grasso diventa lucido.  

Lavorare i tuorli con il pecorino e il pepe nero fino a ottenere una crema. Aggiungere un po' del grasso del guanciale e mescolare. Aggiungere anche un po' di acqua di cottura della pasta non salata per rendere la crema più morbida.  

Cuocere la pasta in acqua non troppo salata fino a quando è al dente. Scolarla, mantenendo un po' di acqua di cottura nella padella con il guanciale.  

Togliere il guanciale dalla padella e aggiungere la pasta scolata. Unire la crema di tuorli e formaggio alla pasta e amalgamare bene a fuoco spento. Aggiungere il guanciale sopra la pasta una volta impiattata.  

Completa il piatto con pecorino e pepe nero macinato fresco.  

Seguendo questi passaggi, otterrai una deliziosa pasta alla carbonara cremosa e piena di sapore. Buon appetito!

CREDITS FOTO  
Javier Somoza, [CC BY-SA 4.0, via Wikimedia Commons](https://creativecommons.org/licenses/by-sa/4.0)

Prenota il tuo soggiorno a Roma in maniera diretta **risparmia sulle commissioni**

## TEL: +39 06 48 49 17 hotelcressy@gmail.com

[![](https://www.romahotelcressy.it/wp-content/uploads/2023/05/WA-150x150.jpg)](https://api.whatsapp.com/send?phone=393517634333&text=Salve%20Hotel%20Cressi%20Roma,%20vorrei%20chattare%20con%20voi)

![](https://www.romahotelcressy.it/wp-content/uploads/2023/05/WA-150x150.jpg)

Comments are closed.

### Search

### Recent Posts

![](https://www.romahotelcressy.it/wp-content/uploads/2015/06/TripAdvisor-logo-300x51.png)


---------- SOURCE: https://www.ceramicavietrese.it/en/recipes-claudia/a-typical-italian-food--pasta-alla-carbonara-a-typical-recipe-of-rome-tradition-and-italian-tradition/

[![VIETRI CERAMICS - the excellence artisan pottery made in Italy]()  VIETRI CERAMICS - the excellence artisan pottery made in Italy](/en/) 

[0](/en/cart/)  [Login](/en/login/)

* [Italiano](/)
* [English](/en/)
* [Español](/es/)
* [Deutsch](/de/)
* [Français](/fr/)
* [Pусский](/ru/)
* [中国](/c2/)

* [Home](/)
* [About us](#) 
  + [Vietri and the ceramic](/en/vietri-and-the-ceramic/)
  + [Our master potters](/en/our-master-potters/)
  + [The art of manufacturing ceramic](/en/the-art-of-manufacturing-ceramic/)
  + [Our design furniture](/en/our-design-furniture/)
  + [Photo Gallery](/en/photo-gallery/)
  + [Video-Gallery](/en/video-gallery/)
  + [Shipments](/en/shipments/)
  + [Payment methods](/en/payment-methods/)
  + [Warranty, defects and refunds](/en/warranty-defects-and-refunds/)
  + [Sure because we have MOCA certification](/en/sure-because-we-have-moca-certification/)
  + [FAQ Frequently Asked Questions](/en/faq-frequently-asked-questions/)
  + [Latest News](/en/news/)
  + [Recipes Claudia](/en/recipes-claudia/)
* [Products Products](#) 
  + [Complete catalog](/en/products/)

    [Prompt Delivery](/en/products/?filter=stock)

    [Complete catalog](/en/products/)  [Prompt Delivery](/en/products/?filter=stock)

     [![Spice Jar with Royal House Decor 2]()](/en/products/spice-jar-with-royal-house-decor-2/?cat=ceramic-jars)

    ### [On the table and in the kitchen](/en/products/on-the-table-and-in-the-kitchen/)

    - [Dishes](/en/products/dishes/)
    - [Chef and Gourmet Shapes Dishes](/en/products/chef-and-gourmet-shapes-dishes/)
    - [PizzaPiatto®](/en/products/pizzapiatto/)
    - [Breakfast](/en/products/breakfast/)
    - [Lunch Dinner](/en/products/lunch-dinner/)
    - [Bowls, Trays and Serving Plates](/en/products/bowls-trays-and-serving-plates/)
    - [Center Table](/en/products/center-table/)
    - [Accessories](/en/products/accessories/)
    - [Ceramic jars](/en/products/ceramic-jars/)
    - [Terrine da Forno](/en/products/terrine-da-forno/)
    - [Trays in wood and tiles](/en/products/trays-in-wood-and-tiles/)

    ### [Wedding favors](/en/products/wedding-favors/)

    - [Wedding](/en/products/wedding/)
    - [Baptism](/en/products/baptism/)
    - [Birthday, Anniversary, Graduation](/en/products/birthday-anniversary-graduation/)
    - [First Holy Communion](/en/products/first-holy-communion/)
    - [Magnets](/en/products/magnets/)

    ### [Home furnishing](/en/products/home-furnishing/)

    - [Bathroom furniture](/en/products/bathroom-furniture/)
    - [Vasi e Cachepot](/en/products/vasi-e-cachepot/)
    - [Wall Plates](/en/products/wall-plates/)
    - [Umbrella stand](/en/products/umbrella-stand/)
    - [Lumi e Lampshade](/en/products/lumi-e-lampshade/)
    - [Objects](/en/products/objects/)
    - [Tiles, Tiles, Floors, Coverings and Kitchen Countertops](/en/products/tiles-tiles-floors-coverings-and-kitchen-countertops/)
    - [Panels - Murals made of project](/en/products/panels--murals-made-of-project/)
    - [Tables and Coffee Tables](/en/products/tables-and-coffee-tables/)

    ### [Garden furniture](/en/products/garden-furniture/)

    - [Kitchen Tables, Coffee Tables, Bench and Outdoor Fountain](/en/products/kitchen-tables-coffee-tables-bench-and-outdoor-fountain/)
    - [Chairs and Stools](/en/products/chairs-and-stools/)
    - [Fountains](/en/products/fountains/)

    ### [Design furniture](/en/products/design-furniture/)

    - [Bar tables and stools](/en/products/bar-tables-and-stools/)
    - [Tables and Coffee Tables](/en/products/tables-and-coffee-tables1/)
    - [Benches and Benches](/en/products/benches-and-benches/)

    ### [Festivity](/en/products/festivity/)

    - [Christmas](/en/products/christmas/)
    - [Easter](/en/products/easter/)

    ### [Artistic](/en/products/artistic/)

    In this section you will find High Ceramics objects, UNIQUE PIECES created by our. Master ...

    ### [Outlet - Discounts up to 60%](/en/products/outlet-discounts-up-to-60/)

    Sales: Here you can find items DISCOUNTED up to 60% end of series or ex display or with small ...
* [Gift Card](/en/gift-card/)
* [Lists  Gifts](#) 
  + [Wedding list](/en/lists-online/wedding-list/)
  + [Gifts list](/en/lists-online/gifts-list/)
  + [Create a list](/en/login/?from=lc)
  + [Search a wedding list](/en/search-for-a-gift-list/?cat=wedding-list)
  + [Search for a gift list](/en/search-for-a-gift-list/?cat=gifts-list)
  + [Give a Gift Card](/en/gift-card/)
* [Contacts](/en/contacts/)

* [Italiano](/)
* [English](/en/)
* [Español](/es/)
* [Deutsch](/de/)
* [Français](/fr/)
* [Pусский](/ru/)
* [中国](/c2/)

[0](/en/cart/ "Cart")

* [Home](/)
* [Recipes Claudia](/en/recipes-claudia/)
* A typical Italian Food - ...

## Recipes Claudia

[Back to the list](/en/recipes-claudia/)

# A typical Italian Food - Pasta alla "Carbonara" a typical recipe of Rome tradition and Italian tradition

A typical recipe of Italian traditions , especially Roman traditions . Easy to do, but that once tasted you can hardly forget . Tradition would like to use only seasoned and grated Pecorino Romano, but if you do not like the taste too strong , you can replace a portion of cheese with Original Parmigiano Reggiano (or original Grana Padano ) that definitely has a taste and a flavor less "strong " .

[![image](/files/admin/immagini/pasta-alla-carbonara-ricetta-tipica-in-ceramica-di-vietri-ceramica-vietrese4-l-20200910230002.jpg)](/files/admin/immagini/pasta-alla-carbonara-ricetta-tipica-in-ceramica-di-vietri-ceramica-vietrese4-l-20200910230002.jpg "Pork cheek") 

Pork cheek

Recipe for 4 people

***Ingredients:***

- 350 gr. of spaghetti of Gragnano

- 3 eggs + 1 egg yolk

- 160gr of cheek lard

- Roman pecorino

- Roman parmisan

- salt and pepper q.b.

***Preparazione:***

Put a pot with abundant water on the fire and when it is going to boile, you'll salt. Use less salt as usual because pasta have already a tasty condiment with cheek lard, parmesan and pecorino.

While you cut the cheek lard in cubes or strips, put in a pot on fire until the fatty become transparent and lightly crounch, then put off from the fire and let it lightly cold.

Break the eggs in a bowl and mix them. Add the pecorino, parmesan and the pepper (as you like) and mix to obtain an homogeneus mixture. Add the cheek lard.

Put the pasta to cook.

When pasta will be ready, dry it and put in the bowl and mix with the mixture and amalgamate them.

***Serve Spaghetti alla Carbonara immediately, adding a fistful of pecorino and black pepper.... and if you used the dish of CeramicaVietrese.it you'll se that they will be better if they are accompanied with art and colors of Vietri ceramic!!!***

[![MeTour.it](/tpl/default/assets/images/metour-banner-2025.jpg)](https://www.metour.it/it)

##### Vietri Ceramic

* [Vietri and the ceramic](/en/vietri-and-the-ceramic/)
* [Our master potters](/en/our-master-potters/)
* [The art of manufacturing ceramic](/en/the-art-of-manufacturing-ceramic/)
* [Our design furniture](/en/our-design-furniture/)
* [Photo Gallery](/en/photo-gallery/)
* [Video-Gallery](/en/video-gallery/)
* [Latest News](/en/news/)
* [Recipes Claudia](/en/recipes-claudia/)
* [Partner Area](/en/login-partner/)

##### Our Shop

* [Products Catalogue](/en/products/)
* [Shipments](/en/shipments/)
* [Payment methods](/en/payment-methods/)
* [Warranty, defects and refunds](/en/warranty-defects-and-refunds/)
* [Sure because we have MOCA certification](/en/sure-because-we-have-moca-certification/)
* [FAQ Frequently Asked Questions](/en/faq-frequently-asked-questions/)

##### Lists online

* [Wedding list](/en/lists-online/wedding-list/)
* [Gifts list](/en/lists-online/gifts-list/)
* [Create a list](/en/login/?from=lc)
* [Search a wedding list](/en/search-for-a-gift-list/?cat=wedding-list)
* [Search for a gift list](/en/search-for-a-gift-list/?cat=gifts-list)

##### Gift Card

* [Give a Gift Card](/en/gift-card/)
* [Ceramica Vietrese Card](/en/gift-card/ceramica-vietrese-card/)
* [Mother's Day Card](/en/gift-card/mothers-day-card/)
* [Happy Birthday Card](/en/gift-card/happy-birthday-card/)
* [Merry Christmas Card](/en/gift-card/merry-christmas-card/)
* [Happy Easter Card](/en/gift-card/happy-easter-card/)
* [Wedding Gift Card](/en/gift-card/wedding-gift-card/)

##### Customer Area

* [User Login](/en/login/)
* [Not registered?](/en/register-on-our-site/)

##### Informations

* [Terms of sale](/en/terms-of-sale/)
* [Customer rights](/en/customer-rights/)
* [Legal notes](/en/legal-notes/)
* [Cookie policy](/en/cookie-policy/)
* [Privacy policy](/en/privacy-policy/)

##### Follow us on

[![VK]()](http://vk.com/id344442374)

##### Subscribe our newsletter

You will receive all updated news directly through your email account.

##### Contacts

Ceramica Vietrese di  
Di Palma Claudia  
VAT 09299191214

 [Contact us](/en/contacts/)

[![Italiano]( "Italiano")](/)  [![English]( "English")](/en)  [![Español]( "Español")](/es)  [![Deutsch]( "Deutsch")](/de)  [![Français]( "Français")](/fr)  [![Pусский]( "Pусский")](/ru)  [![中国]( "中国")](/c2)

© 2026 Di Palma Claudia   2015 website by [dynform.it](http://www.dynform.it/)   
& [studio150.it](http://www.studio150.it/)

# DOCUMENT LOADING

In [16]:
import json
from langchain_core.documents import Document

tool_results = json.loads(response["messages"][-2].content).get("results")

docs=[]
for result in tool_results:
    docs.append(
        Document(
            page_content = result.get("raw_content"),
            metadata = {"url": result.get("url"), "title": result.get("title")},
        )
    )

###### ONLY FOR DEBUG PURPOSES

In [17]:
for doc in docs:
    print(doc.metadata["url"])
    print(f"{doc.page_content[:400]}\n\n\n\n")    

https://www.romahotelcressy.it/la-pasta-alla-carbonara/
![it](https://www.romahotelcressy.it/wp-content/plugins/sitepress-multilingual-cms/res/flags/it.png)
![en](https://www.romahotelcressy.it/wp-content/plugins/sitepress-multilingual-cms/res/flags/en.png)
![Roma Hotel Cressy](https://www.romahotelcressy.it/wp-content/uploads/2020/09/HC-h95.jpg)

### Main menu

### LA PASTA ALLA CARBONARA

![](https://www.romahotelcressy.it/wp-content/uploads/2024/04/




https://www.ceramicavietrese.it/en/recipes-claudia/a-typical-italian-food--pasta-alla-carbonara-a-typical-recipe-of-rome-tradition-and-italian-tradition/
[![VIETRI CERAMICS - the excellence artisan pottery made in Italy]()  VIETRI CERAMICS - the excellence artisan pottery made in Italy](/en/) 

[0](/en/cart/)  [Login](/en/login/)

* [Italiano](/)
* [English](/en/)
* [Español](/es/)
* [Deutsch](/de/)
* [Français](/fr/)
* [Pусский](/ru/)
* [中国](/c2/)

* [Home](/en/)
* [About us](#) 
  + [Vietri and the ceramic](/en/vietri-and-the-ceram

# DOCUMENT SPLITTING
[https://docs.langchain.com/oss/python/integrations/splitters/markdown_header_metadata_splitter]

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=120,
    separators=[r"\n## ", r"\n### ", r"\n#### ", r"\n\n", r"\n", r"(?<=\. )", " ", ""], 
    is_separator_regex=True
)

raw_splits = r_splitter.split_documents(docs)

splits = []
for split in raw_splits:
    allowed_meta = ["url", "title"]
    meta_parts = []
    for key in allowed_meta:
        if key in split.metadata:
            value = split.metadata[key]
            meta_parts.append(f"{key}: {value}")
    
    metadata_header = f"[{', '.join(meta_parts)}]\n"
    split.page_content = metadata_header + split.page_content
    splits.append(split)    

###### ONLY FOR DEBUG PURPOSES

In [19]:
_ = [print(split.metadata["url"] + "\n" + split.metadata["title"] + "\n" + split.page_content + "\n\n") for split in splits]

https://www.romahotelcressy.it/la-pasta-alla-carbonara/
CARBONARA PASTA - Roma Hotel Cressy
[url: https://www.romahotelcressy.it/la-pasta-alla-carbonara/, title: CARBONARA PASTA - Roma Hotel Cressy]
![it](https://www.romahotelcressy.it/wp-content/plugins/sitepress-multilingual-cms/res/flags/it.png)
![en](https://www.romahotelcressy.it/wp-content/plugins/sitepress-multilingual-cms/res/flags/en.png)
![Roma Hotel Cressy](https://www.romahotelcressy.it/wp-content/uploads/2020/09/HC-h95.jpg)

### Main menu


https://www.romahotelcressy.it/la-pasta-alla-carbonara/
CARBONARA PASTA - Roma Hotel Cressy
[url: https://www.romahotelcressy.it/la-pasta-alla-carbonara/, title: CARBONARA PASTA - Roma Hotel Cressy]
### LA PASTA ALLA CARBONARA

![](https://www.romahotelcressy.it/wp-content/uploads/2024/04/Espaguetis_carbonara.jpeg)

# LA PASTA ALLA CARBONARA

LA STORIA


https://www.romahotelcressy.it/la-pasta-alla-carbonara/
CARBONARA PASTA - Roma Hotel Cressy
[url: https://www.romahotelcressy.it/la-pa

# DOCUMENT EMBEDDINGS

In [20]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os

current_directory = os.getcwd()

ids = ["id" + str(index + 1) for index, val in enumerate(splits)]

vector_store = Chroma(
    collection_name="webpages_content_collection",
    embedding_function=HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2"),
    #persist_directory=current_directory, 
)

vector_store.add_documents(
    documents = splits,
    #ids = ids,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['id1',
 'id2',
 'id3',
 'id4',
 'id5',
 'id6',
 'id7',
 'id8',
 'id9',
 'id10',
 'id11',
 'id12',
 'id13',
 'id14',
 'id15',
 'id16',
 'id17',
 'id18',
 'id19',
 'id20',
 'id21',
 'id22',
 'id23',
 'id24',
 'id25',
 'id26',
 'id27',
 'id28',
 'id29',
 'id30',
 'id31',
 'id32',
 'id33']

# RAG-BASED CONVERSATION

In [21]:
query = ("Quanto pepe devo macinare per fare una carbonara perfetta?")

result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    {"configurable": {"thread_id": "1"}},
)

result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Basato sulle informazioni estratte dai siti web, **nessuna quantità esatta di pepe è specificata** per la pasta alla carbonara. Entrambe le ricette usano la notazione:

- **"Pepe nero q.b."** (quantità bastevole)
- **"Add the pepper (as you like)"** (aggiungi il pepe come preferisci)

Tuttavia, ecco alcune indicazioni pratiche:

**Per una carbonara perfetta:**

1. **Tempismo importante:** Il pepe viene aggiunto **insieme alle uova** quando prepari la crema prima di unirla alla pasta
2. **Quantità consigliata:** Generalmente si usa **1/2 - 1 cucchiaino di pepe nero macinato fresco** per 4 persone
3. **Fresco è essenziale:** Il pepe deve essere **macinato al momento**, non usare pepe in grani vecchio
4. **Al momento del servizio:** Molti aggiungono un **extra di pepe nero fresco** sopra il piatto prima di servire

Il pepe serve a:
- **Equilibrare** la sapidità del guanciale, pecorino e parmigiano
- **Dare p

In [22]:
query = ("Quanto marsala è richiesto?")

result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    {"configurable": {"thread_id": "1"}},
)

result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Sulla base delle informazioni estratte dai siti web che ho analizzato, **il marsala NON è richiesto nella ricetta della carbonara**.

**Ecco cosa dicono le fonti:**

---

### 1. **Roma Hotel Cressy**
**Ingredienti richiesti:**
- Spaghetti 320 g
- Tuorli 6
- Pepe nero q.b.
- Guanciale 150 g
- Pecorino romano 50 g

**Niente marsala.**

---

### 2. **Ceramica Vietrese**
**Ingredienti richiesti:**
- 350 gr di spaghetti di Gragnano
- 3 uova + 1 tuorlo
- 160 gr di guanciale
- Pecorino romano
- Parmigiano
- Sale e pepe q.b.

**Niente marsala.**

---

## ⚠️ **Importante da sapere:**

La **carbonara è un piatto romano tradizionale** e la ricetta autentica **NON include**:
- ❌ Vino (neppure marsala)
- ❌ Panna
- ❌ Cipolle
- ❌ Funghi
- ❌ Altri formaggi

L'aggiunta di **panna, vino o altri liquidi** è considerato da molti **tradizionisti una "mancanza"** alla vera carbonara.

---

## **Perché non si usa marsala?**

Il

In [23]:
query = ("Mi sono confuso con il marsala. Tornando al pepe, quali fonti hai consultato? Scrivimi sia i titoli sia gli url")

result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    {"configurable": {"thread_id": "1"}},
)

result["messages"][-1].pretty_print()

================================== Ai Message ==================================

### Fonti Consultate per le Informazioni sulla Carbonara

Ho consultato **2 siti web** con estrazione completa del contenuto:

---

## 📄 **Sito 1**
**Titolo:** LA PASTA ALLA CARBONARA - Roma Hotel CressyRoma Hotel Cressy
**URL:** https://www.romahotelcressy.it/la-pasta-alla-carbonara/
**Contenuto estratto:** ✅ Completo (storia, ingredienti, preparazione)

---

## 📄 **Sito 2**
**Titolo:** Pasta alla "Carbonara" a typical recipe of Rome tradition and Italian tradition
**URL:** https://www.ceramicavietrese.it/en/recipes-claudia/a-typical-italian-food--pasta-alla-carbonara-a-typical-recipe-of-rome-tradition-and-italian-tradition/
**Contenuto estratto:** ✅ Completo (ingredienti, preparazione in inglese/italiano)

---

## 📋 **Sito 3 (solo ricerca, non estrazione)**
**Titolo:** Carbonara Spaghetti the queen of the city of Rome - Italiaregina.it
**URL:** https://www.italiaregina.it/carbonara-spaghetti/
**Contenut

# ELAPSED TIME

In [24]:
stop_time = time.perf_counter()

elapsed_time = stop_time - start_time

minutes = int(elapsed_time // 60)
seconds = elapsed_time % 60

print(f"Execution time: {minutes:02d}:{seconds:05.2f}")

Execution time: 64:08.78
